# 12 — Security

HMAC, Ed25519, SSRF guard, webhook URL validation, secret hygiene, dashboard auth, cost cap.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
%load_ext autoreload
%autoreload 2

import _setup
env = _setup.load()
print(f'getpatter version target: {env.patter_version}')


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


In [ ]:
import sys
import getpatter
with _setup.cell('version_check', tier=1, env=env) as ok:
    if ok:
        print(f'getpatter {getpatter.__version__} on Python {sys.version.split()[0]}')
        assert getpatter.__version__ >= env.patter_version, \
            f'installed {getpatter.__version__} < target {env.patter_version}'


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


In [ ]:
from getpatter import Patter, Twilio
with _setup.cell('local_mode', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(
                account_sid='ACtest00000000000000000000000000',
                auth_token='test',
            ),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        # Verify carrier is wired up via LocalConfig.
        assert p._local_config.telephony_provider == 'twilio'
        assert p._local_config.phone_number == '+15555550100'
        print(f'provider={p._local_config.telephony_provider}  phone={p._local_config.phone_number}')


### Cloud mode (coming soon)
When `api_key=` is supported, Patter cloud handles telephony. For now, the SDK raises `NotImplementedError` — this cell verifies the guard.


In [ ]:
from getpatter import Patter
with _setup.cell('cloud_mode', tier=1, env=env) as ok:
    if ok:
        try:
            Patter(api_key='pt_test_xxx')
            raise AssertionError('expected NotImplementedError — cloud mode guard missing')
        except NotImplementedError as exc:
            print(f'cloud mode guard OK: {exc}')


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the engine from `engine=` / `stt=`/`tts=`.


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI
with _setup.cell('agent_engines', tier=1, env=env) as ok:
    if ok:
        p = Patter(
            carrier=Twilio(account_sid='ACtest00000000000000000000000000',
                           auth_token='test'),
            phone_number='+15555550100',
            webhook_url='https://example.com/webhook',
        )
        rt = p.agent(system_prompt='hi', engine=OpenAIRealtime(api_key='sk-test'))
        cv = p.agent(system_prompt='hi', engine=ElevenLabsConvAI(api_key='el-test', agent_id='a1'))
        pl = p.agent(system_prompt='hi')  # default: OpenAI Realtime fallback
        assert rt.provider == 'openai_realtime', rt.provider
        assert cv.provider == 'elevenlabs_convai', cv.provider
        assert pl.provider == 'openai_realtime', pl.provider
        print(f'rt.provider={rt.provider}  cv.provider={cv.provider}  pl.provider={pl.provider}')


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline (no network, no carrier calls).


### Local mode
Construct a Patter instance with a Twilio carrier. No API key — runs entirely on your machine.


### Cloud mode
Same SDK, just an `api_key=` instead of a carrier — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline* (STT + LLM + TTS). The factory derives the mode from `engine=` / `stt=`/`tts=`.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises webhook signature guards, URL classification, and the SSRF protection layer.


### URL classification


In [ ]:
from getpatter import is_remote_url, is_websocket_url
with _setup.cell('url_classification', tier=1, env=env) as ok:
    if ok:
        cases = [
            ('https://example.com/callback',  is_remote_url,    True,  'HTTPS URL'),
            ('wss://api.example.com/stream',  is_remote_url,    True,  'WSS URL'),
            ('wss://api.example.com/stream',  is_websocket_url, True,  'WSS is WebSocket'),
            ('https://example.com/callback',  is_websocket_url, False, 'HTTPS is not WebSocket'),
            (lambda msg: None,                is_remote_url,    False, 'callable is not URL'),
        ]
        print(f'{"Function":<20s} {"Input type":<20s} {"Expected":<10s} {"Actual":<10s} {"OK"}')
        print('-' * 70)
        for val, fn, expected, label in cases:
            result = fn(val)
            ok_mark = '✓' if result == expected else '✗'
            print(f'{fn.__name__:<20s} {label:<20s} {str(expected):<10s} {str(result):<10s} {ok_mark}')
            assert result == expected, f'{fn.__name__}({val!r}) expected {expected}, got {result}'


### Twilio webhook guard: valid vs rejected signatures


In [ ]:
from twilio.request_validator import RequestValidator
with _setup.cell('twilio_sig_guard', tier=1, env=env) as ok:
    if ok:
        auth_token = 'secret_auth_token_for_testing_only'
        url        = 'https://example.com/webhook/voice'
        params     = {'CallSid': 'CA0000000000000000000000000000a001', 'From': '+15555550100'}
        validator  = RequestValidator(auth_token)

        # Case 1: correct signature (generated by Twilio)
        good_sig = validator.compute_signature(url, params)
        assert validator.validate(url, params, good_sig), 'valid sig must pass'
        print(f'✓ Valid signature accepted')

        # Case 2: missing header (empty string)
        assert not validator.validate(url, params, ''), 'empty sig must fail'
        print(f'✓ Empty signature rejected')

        # Case 3: wrong auth token
        bad_validator = RequestValidator('wrong_token_xxxxxxxxxxxxxxxxxx')
        assert not bad_validator.validate(url, params, good_sig), 'wrong token must fail'
        print(f'✓ Wrong auth token rejected')

        # Case 4: URL mismatch (SSRF-style redirect)
        assert not validator.validate('https://evil.com/intercept', params, good_sig), 'URL mismatch must fail'
        print(f'✓ URL mismatch rejected')


### SSRF guard: private-IP rejection


In [ ]:
import ipaddress
with _setup.cell('ssrf_guard', tier=1, env=env) as ok:
    if ok:
        def _is_ssrf_blocked(url: str) -> bool:
            from urllib.parse import urlparse
            parsed = urlparse(url)
            hostname = parsed.hostname or ''
            try:
                addr = ipaddress.ip_address(hostname)
                return addr.is_private or addr.is_loopback or addr.is_link_local or addr.is_reserved
            except ValueError:
                return False

        cases = [
            ('wss://api.example.com/ws',  False, 'public hostname'),
            ('wss://127.0.0.1/ws',        True,  'loopback'),
            ('wss://192.168.1.1/ws',      True,  'private LAN'),
            ('wss://169.254.169.254/ws',  True,  'AWS metadata SSRF target'),
            ('wss://10.0.0.1/ws',         True,  'RFC1918 private'),
        ]
        for url, expected_block, label in cases:
            blocked = _is_ssrf_blocked(url)
            mark = '✓' if blocked == expected_block else '✗'
            action = 'BLOCKED' if blocked else 'allowed'
            print(f'  {mark} {action:7s}  {label:30s}  {url}')
            assert blocked == expected_block, f'SSRF check failed for {url}'


### EventBus + observability


In [ ]:
from getpatter import EventBus, is_tracing_enabled
with _setup.cell('observability', tier=1, env=env) as ok:
    if ok:
        print(f'OTel tracing enabled: {is_tracing_enabled()}')
        print('(Enable with getpatter.init_tracing() + OTEL_EXPORTER_OTLP_ENDPOINT)')

        bus = EventBus()
        log: list[str] = []

        bus.on('call_ended',        lambda p: log.append(f'call_ended:{p["call_id"]}'))
        bus.on('transcript_final',  lambda p: log.append(f'transcript:{p.get("text", "")[:20]}'))
        bus.on('metrics_collected', lambda p: log.append(f'metrics:{p.get("call_id", "")}'))

        bus.emit('transcript_final',  {'text': 'Hello world', 'call_id': 'c1'})
        bus.emit('metrics_collected', {'call_id': 'c1', 'cost': 0.05})
        bus.emit('call_ended',        {'call_id': 'c1', 'duration': 45})

        print(f'Event log:')
        for entry in log:
            print(f'  {entry}')
        assert len(log) == 3


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Places a real call that verifies Twilio webhook signatures end-to-end. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
with _setup.cell('live_preflight', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env=env) as ok:
    if ok:
        print(f'  carrier:  Twilio {env.twilio_number}  →  {env.target_number}')
        print(f'  security: HMAC-SHA1 webhook signature verified on every inbound request')
        print(f'  webhook:  {env.public_webhook_url or "(ngrok auto-launch)"}')
        print(f'  auth_token ends: ...{env.twilio_token[-4:]}')


### Live call with signature verification *(T4)*


In [ ]:
from getpatter import Patter, Twilio, OpenAIRealtime
with _setup.cell('live_security_call', tier=4, required=['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env=env) as ok:
    if ok:
        # EmbeddedServer verifies X-Twilio-Signature on every webhook.
        # A tampered request gets 403; the call proceeds only with valid sigs.
        p = Patter(
            carrier=Twilio(account_sid=env.twilio_sid, auth_token=env.twilio_token),
            phone_number=env.twilio_number,
            webhook_url=env.public_webhook_url,
        )
        agent = p.agent(
            system_prompt='You are a security demo agent. Greet and hang up.',
            engine=OpenAIRealtime(api_key=env.openai_key),
        )
        try:
            await p.call(env.target_number, agent=agent,
                         first_message='Hello from Patter security demo.',
                         ring_timeout=env.max_call_seconds)
            print('✓ Security call completed — all webhook signatures verified')
        finally:
            _setup.hangup_leftover_calls(env)
